# Cap and caplet valuation

Value a single SOFR caplet and a quarterly cap strip using the Black pricers.
Both curves and a Black volatility surface are required. Follow
[setup](../../README.md#install-and-run), then run every cell in order.
Inputs use the repository's illustrative 2026-02-24 market.

In [1]:
import pandas as pd
import quantsupport as qs
from market_data import load_market

market = load_market()
ref = market.quotes.reference_date


## A caplet and a strip of caplets

The caplet pays `notional × accrual × max(SOFR − strike, 0)` at the end of
one future accrual period. Fixing, accrual start, accrual end, and payment date
are separate terms even when some dates coincide. A `CapFloor` builds a strip
of these periods from a frequency.

The Python binding dispatches these products to the Black closed-form pricers.
The SOFR curve determines the forward and discount factor; the SOFR Black
surface provides volatility at the option expiry and strike. Attaching the
surface is necessary even though it was unnecessary for the swap and bond.

In [2]:
fixing = ref + "1Y"
payment = fixing + "3M"
caplet = qs.CapletFloorlet(
    identifier="USD-CAPLET-1Y",
    fixing_date=fixing,
    start_accrual_date=fixing,
    end_accrual_date=payment,
    payment_date=payment,
    strike=0.045,
    currency=qs.Currency.USD,
    market_index=qs.MarketIndex.SOFR,
    payoff_type=qs.CapletFloorletType.Caplet,
)
cap = qs.CapFloor(
    identifier="USD-CAP-1Y-3Y",
    start_date=fixing,
    maturity_date=ref + "3Y",
    strike=0.045,
    notional=1_000_000,
    currency=qs.Currency.USD,
    market_index=qs.MarketIndex.SOFR,
    cap_floor_type=qs.CapFloorType.Cap,
    frequency=qs.Frequency.Quarterly,
)
context = qs.PricingContext(
    quotes=market.quotes,
    curves=market.curves,
    csa_policy=market.csa_policy,
    volatility_surfaces=market.volatility_surfaces,
)

with context:
    caplet_result = context.evaluate(
        qs.Trade(caplet, ref, notional=1_000_000),
        [qs.Request.Value, qs.Request.Sensitivities],
    )
    cap_result = context.evaluate(
        qs.Trade(cap, ref),
        [qs.Request.Value, qs.Request.Sensitivities],
    )

pd.DataFrame(
    [
        {"product": "single caplet", "pv_usd": caplet_result.price},
        {"product": "quarterly cap strip", "pv_usd": cap_result.price},
    ]
)

,product,pv_usd
0,single caplet,834.330964
1,quarterly cap strip,6067.398522


The caplet risk contains both interest-rate and volatility quote labels. A
volatility quote of `0.30` is a 30% Black volatility; a +1 volatility-point move
is `0.01`, whereas +1 bp on a rate is `0.0001`. Do not scale the whole table by
the same number and call it DV01. The next notebook separates these shocks.

In [3]:
caplet_result.sensitivities.sort_values(
    "value",
    key=lambda series: series.abs(),
    ascending=False,
).head(10)


,pillar,value
30,OIS_USD_SOFR_2Y,196958.793948
29,OIS_USD_SOFR_1Y,-95220.527119
7,CapletFloorlet_USD_SOFR_3M_1Y_Absolute_0.045_S...,3888.965205
27,FixedRateDeposit_USD_SOFR_1D,-2.317296
25,CapletFloorlet_USD_SOFR_3M_7Y_Absolute_0.045_S...,0.000000
20,CapletFloorlet_USD_SOFR_3M_5Y_Absolute_0.055_S...,0.000000
21,CapletFloorlet_USD_SOFR_3M_6M_Absolute_0.035_S...,0.000000
22,CapletFloorlet_USD_SOFR_3M_6M_Absolute_0.045_S...,0.000000
23,CapletFloorlet_USD_SOFR_3M_6M_Absolute_0.055_S...,0.000000
24,CapletFloorlet_USD_SOFR_3M_7Y_Absolute_0.035_S...,0.000000


In [4]:
assert caplet_result.price > 0
assert cap_result.price > caplet_result.price
assert not caplet_result.sensitivities.empty
print("Both option values are positive; the cap strip contains multiple caplets.")

Both option values are positive; the cap strip contains multiple caplets.


Useful cap/caplet requests are `Value` and `Sensitivities`. Black volatility
quote risk and rate quote risk use different bump sizes. Continue with
[curves and volatility](../../volatilitysurface/curves_and_volatility.ipynb)
for independent market shocks.